In [ ]:
# =============================================================================
#  ECG ARRHYTHMIA CLASSIFICATION — RAW vs SPECTRAL GATING + LSTM
#  3-CLASS VERSION: Normal / Atrial / Other
#
#  KIẾN TRÚC TỔNG QUAN:
#    Input  : RR-interval features per beat  (RR, BPM, RR_z) + raw beat waveform
#    Phase 1: RAW + Multi-input BiLSTM classifier (baseline)
#             → dùng waveform raw (đã baseline+bandpass), RR-seq, handcrafted feats
#             → tại test: tính log-likelihood của RR-seq dưới mỗi class-model
#                        → predict class có likelihood cao nhất
#             (đây là baseline trực tiếp để so sánh với STL-denoised)
#
#    Phase 2: Spectral-denoised (beat-level) → Multi-input BiLSTM classifier
#    Phase 3: Spectral-denoised (record-level) → Multi-input BiLSTM classifier
#             → chạy STL trên tín hiệu dài theo từng record
#             → sau đó mới cắt beat 180 mẫu quanh R-peak
#                (các thành phần không tự-tương-quan = nhiễu, bị model loại bỏ)
#             → Đưa STL-denoised waveform vào BiLSTM classifier
#             → Có thể so sánh trực tiếp với LSTM-DAE codebase #1
#
#  TẠI SAO SPECTRAL GATING CHO PIPELINE NÀY?
#    - STL cần chuỗi đủ dài để seasonal/trend decomposition có ý nghĩa
#    - Phần residual (signal − fitted) chính là nhiễu trắng (white noise)
#    - → Fitted values = denoised version
#    - So sánh công bằng: cùng classifier, chỉ khác bước denoising (RAW vs STL)
#    - Điều này phản ánh trực tiếp lợi ích của STL trong pipeline phân loại
#                        nhưng chậm hơn và yếu hơn với non-linearity
# =============================================================================

import os, warnings, gc, random
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as sps
from scipy.signal import medfilt, find_peaks, stft, istft
from scipy.stats import skew, kurtosis
import pywt
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, fbeta_score, accuracy_score,
                             precision_score, recall_score,
                             roc_auc_score, roc_curve, auc, confusion_matrix,
                             classification_report)
from imblearn.over_sampling import SMOTE
from tqdm import tqdm

# Time-series and deep-learning stack

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED)

import tensorflow as tf
from tensorflow.keras import layers, Model, Input, callbacks, optimizers
from tensorflow.keras.utils import to_categorical
tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        try:
            tf.config.experimental.set_memory_growth(gpu, True)
        except Exception:
            pass
    print(f"TensorFlow GPU enabled: {len(gpus)} device(s)")
else:
    print("TensorFlow GPU not found, using CPU")

plt.rcParams.update({
    'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'font.size': 9,
})

# =============================================================================
# CONFIG
# =============================================================================
EXTRACT_DIR = '/kaggle/input/datasets/mondejar/mitbih-database'
FS          = 360
WB, WA      = 90, 90
BEAT_LEN    = WB + WA       # 180

LABEL_MAP = {
    'N': 'Normal', 'L': 'Normal', 'R': 'Normal', 'e': 'Normal', 'j': 'Normal',
    'A': 'Atrial', 'a': 'Atrial', 'J': 'Atrial', 'S': 'Atrial',
    'V': 'Other',  'E': 'Other',  'F': 'Other',
    '/': 'Other',  'f': 'Other',  'Q': 'Other',
}
CLASSES_ORDER = ['Normal', 'Atrial', 'Other']
N_CLASSES     = 3

TRAIN_PATIENTS = [100, 101, 103, 105, 106, 108, 109, 111, 112, 115,
                  116, 117, 118, 119, 121, 122, 123, 124, 200, 201,
                  202, 203, 205, 207, 208, 209, 215]
TEST_PATIENTS  = [220, 221, 222, 223, 228, 230, 231, 232, 233]

PALETTE = {
    'P1_RAW_LSTM' : '#6A1B9A',
    'P2_STL_LSTM': '#1565C0',
}

OUT_DIR = '/kaggle/working'
os.makedirs(OUT_DIR, exist_ok=True)


def resolve_mitbih_dir(default_dir):
    candidates = [
        default_dir,
        '/kaggle/input/mitbih-database',
        '/kaggle/input/mit-bih-arrhythmia-database',
        r'G:\My Drive\TKUD\MIT_BIH_Database',
    ]
    for path in candidates:
        if os.path.exists(os.path.join(path, '100.csv')) and os.path.exists(os.path.join(path, '100annotations.txt')):
            return path

    kaggle_root = '/kaggle/input'
    if os.path.exists(kaggle_root):
        for root, _, files in os.walk(kaggle_root):
            files = set(files)
            if '100.csv' in files and '100annotations.txt' in files:
                return root

    raise FileNotFoundError(
        "Cannot find MIT-BIH dataset. Attach the dataset on Kaggle and make sure the folder contains "
        "100.csv and 100annotations.txt. Current default EXTRACT_DIR: " + str(default_dir)
    )

EXTRACT_DIR = resolve_mitbih_dir(EXTRACT_DIR)
print(f"Using MIT-BIH dataset path: {EXTRACT_DIR}")

# Hyperparams
SEQ_LEN      = 10
LSTM_UNITS   = 64
LSTM_DROPOUT = 0.3
LSTM_EPOCHS  = 40
LSTM_BATCH   = 128
LSTM_LR      = 1e-3

# STL settings được định nghĩa ở Phase 2 (record-level denoising)


# =============================================================================
# 1. DATA LOADING
# =============================================================================
def load_record(rid):
    csv_path = os.path.join(EXTRACT_DIR, f'{rid}.csv')
    ann_path = os.path.join(EXTRACT_DIR, f'{rid}annotations.txt')

    sig_df = pd.read_csv(csv_path)
    sig_df.columns = [c.strip().strip("'\"") for c in sig_df.columns]
    col_map = {c.upper(): c for c in sig_df.columns}
    mlii_col = col_map.get('MLII')
    if mlii_col is None:
        num_cols = [c for c in sig_df.columns
                    if sig_df[c].dtype.kind in 'iuf' and 'sample' not in c.lower()]
        mlii_col = num_cols[0] if num_cols else None
    if mlii_col is None:
        raise ValueError(f"No MLII or numeric signal column found in {csv_path}. Columns: {list(sig_df.columns)}")
    mlii = sig_df[mlii_col].interpolate('linear').ffill().bfill().values.astype(float)

    beats_ann = []
    with open(ann_path, 'r') as f:
        next(f)
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3: continue
            try:
                sample_pos = int(parts[1])
                beat_type  = parts[2]
                beats_ann.append((sample_pos, beat_type))
            except ValueError:
                continue
    return mlii, beats_ann, len(mlii)


# =============================================================================
# 2. PREPROCESSING
# =============================================================================
def remove_baseline(sig):
    baseline = medfilt(sig, 71)
    baseline = medfilt(baseline, 215)
    return sig - baseline

def bandpass_filter(x, lo=0.5, hi=40.0, fs=FS, order=4):
    nyq = fs / 2
    b, a = sps.butter(order, [lo/nyq, hi/nyq], btype='band')
    return sps.filtfilt(b, a, np.asarray(x, dtype=float))

def preprocess_signal(sig):
    return bandpass_filter(remove_baseline(sig.astype(float)))


# =============================================================================
# 3. RR / BPM / RR_z
# =============================================================================
def compute_rr_bpm_zscore(r_positions, fs=FS):
    n = len(r_positions)
    if n < 2:
        return np.zeros(n), np.zeros(n), np.zeros(n)
    rr_samples = np.diff(r_positions)
    rr_seconds = rr_samples / fs
    RR  = np.concatenate([[rr_seconds[0]], rr_seconds])
    BPM = 60.0 / np.clip(RR, 0.2, 3.0)
    mu, sd = RR.mean(), RR.std() + 1e-8
    RR_z = (RR - mu) / sd
    return RR, BPM, RR_z


# =============================================================================
# 4. HANDCRAFTED FEATURES
# =============================================================================
def feat_morph(b):
    b = np.asarray(b, dtype=float)
    return [
        b.mean(), b.std(), b.min(), b.max(),
        float(b.argmax()) / len(b),
        float(b.argmin()) / len(b),
        float(skew(b)), float(kurtosis(b)),
        float(np.trapz(np.abs(b))),
        float(np.sum(np.diff(b)**2)),
        float(np.max(np.abs(np.diff(b)))),
        float(np.sum(b > 0)) / len(b),
        float(np.percentile(b, 25)),
        float(np.percentile(b, 75)),
    ]

def feat_wavelet(b, wavelet='db1', level=3):
    coeffs = pywt.wavedec(np.asarray(b, dtype=float), wavelet, level=level)
    feats = []
    for c in coeffs:
        feats += [float(np.sum(c**2)), float(np.abs(c).max()), float(c.mean())]
    return feats

def extract_handcrafted(beat, rr, bpm, rr_z):
    return feat_morph(beat) + feat_wavelet(beat) + [rr, bpm, rr_z]

def sanitize(X):
    X = np.array(X, dtype=float)
    X = np.where(np.isinf(X), np.nan, X)
    df = pd.DataFrame(X)
    return df.interpolate('linear').ffill().bfill().fillna(0).values


# =============================================================================
# 5. BUILD DATASET (Patient-wise split)
# =============================================================================
print("\n" + "="*72)
print("  LOADING DATA — Patient-wise split, 3-class (Normal/Atrial/Other)")
print("="*72)

def build_per_patient_beats(rid):
    try:
        mlii, beats_ann, sig_len = load_record(rid)
    except FileNotFoundError:
        return []
    except Exception as e:
        print(f"  [WARN] {rid}: {e}")
        return []

    sig_clean = preprocess_signal(mlii)
    r_positions, labels = [], []
    for sample_pos, beat_type in beats_ann:
        if beat_type not in LABEL_MAP: continue
        if sample_pos - WB < 0 or sample_pos + WA >= sig_len: continue
        r_positions.append(sample_pos)
        labels.append(LABEL_MAP[beat_type])
    r_positions = np.array(r_positions)
    if len(r_positions) < 2: return []

    RR, BPM, RR_z = compute_rr_bpm_zscore(r_positions)
    out = []
    for i, (s, lab) in enumerate(zip(r_positions, labels)):
        beat = sig_clean[s - WB : s + WA]
        feat = extract_handcrafted(beat, RR[i], BPM[i], RR_z[i])
        out.append({
            'beat': beat.astype(np.float32),
            'rr': float(RR[i]), 'bpm': float(BPM[i]), 'rr_z': float(RR_z[i]),
            'feat': feat, 'label': lab, 'patient': rid,
        })
    return out

def build_dataset():
    train_beats, test_beats = [], []
    print("\n  Loading TRAIN patients ...")
    for rid in tqdm(TRAIN_PATIENTS):
        train_beats += build_per_patient_beats(rid)
    print("\n  Loading TEST patients ...")
    for rid in tqdm(TEST_PATIENTS):
        test_beats += build_per_patient_beats(rid)
    return train_beats, test_beats

train_beats, test_beats = build_dataset()
print(f"\n  Total train beats : {len(train_beats)}")
print(f"  Total test  beats : {len(test_beats)}")

if len(train_beats) == 0 or len(test_beats) == 0:
    raise RuntimeError(
        f"No beats were created. Check EXTRACT_DIR={EXTRACT_DIR}, available record files, and annotation format. "
        f"train_beats={len(train_beats)}, test_beats={len(test_beats)}"
    )


# =============================================================================
# 6. CONVERT TO ARRAYS
# =============================================================================
def beats_to_arrays(beats_list):
    W   = np.array([b['beat']  for b in beats_list], dtype=np.float32)[..., None]
    F   = np.array([b['feat']  for b in beats_list], dtype=np.float32)
    RR  = np.array([[b['rr'], b['bpm'], b['rr_z']] for b in beats_list],
                   dtype=np.float32)
    y   = np.array([b['label'] for b in beats_list])
    pid = np.array([b['patient'] for b in beats_list])
    return W, F, RR, y, pid

X_w_tr, X_f_tr, X_rr_tr, y_tr_raw, pid_tr = beats_to_arrays(train_beats)
X_w_te, X_f_te, X_rr_te, y_te_raw, pid_te = beats_to_arrays(test_beats)

le = LabelEncoder()
le.fit(CLASSES_ORDER)
y_tr_enc = le.transform(y_tr_raw)
y_te_enc = le.transform(y_te_raw)

print(f"\n  Classes     : {list(le.classes_)}")
print(f"  Train dist  : {dict(zip(*np.unique(y_tr_raw, return_counts=True)))}")
print(f"  Test  dist  : {dict(zip(*np.unique(y_te_raw, return_counts=True)))}")


# =============================================================================
# 7. RR SEQUENCE BUILDING (per patient)
# =============================================================================
def build_rr_sequences(X_rr, pid, seq_len=SEQ_LEN):
    N = len(X_rr)
    seqs = np.zeros((N, seq_len, 3), dtype=np.float32)
    for i in range(N):
        cur_pid = pid[i]
        start = max(0, i - seq_len + 1)
        candidate_idx = [j for j in range(start, i + 1) if pid[j] == cur_pid]
        seq = X_rr[candidate_idx]
        if len(seq) < seq_len:
            pad = np.repeat(seq[:1], seq_len - len(seq), axis=0)
            seq = np.vstack([pad, seq])
        seqs[i] = seq[-seq_len:]
    return seqs

print("\n  Building RR sequences (10-beat windows per patient) ...")
X_seq_tr = build_rr_sequences(X_rr_tr, pid_tr)
X_seq_te = build_rr_sequences(X_rr_te, pid_te)
print(f"  Sequence shape — train: {X_seq_tr.shape}, test: {X_seq_te.shape}")


# =============================================================================
# 8. NORMALIZATION
# =============================================================================
def zscore_waveform(W):
    mu = W.mean(axis=1, keepdims=True)
    sd = W.std(axis=1, keepdims=True) + 1e-8
    return (W - mu) / sd

X_w_tr_z = zscore_waveform(X_w_tr)
X_w_te_z = zscore_waveform(X_w_te)

sc_f = StandardScaler()
X_f_tr_s = sc_f.fit_transform(X_f_tr)
X_f_te_s = sc_f.transform(X_f_te)

sc_rr = StandardScaler()
N_tr, T_seq, C = X_seq_tr.shape
X_seq_tr_s = sc_rr.fit_transform(X_seq_tr.reshape(-1, C)).reshape(N_tr, T_seq, C)
X_seq_te_s = sc_rr.transform(X_seq_te.reshape(-1, C)).reshape(len(X_seq_te), T_seq, C)


# =============================================================================
# 9. PHASE 1 — RAW + Multi-Input BiLSTM Classifier
# =============================================================================
print("\n" + "="*72)
print("  PHASE 1 — RAW + Multi-Input BiLSTM Classifier")
print("="*72)

def apply_smote_multi(X_w, X_f, X_seq, y):
    Nw, T_w, _ = X_w.shape
    N_, T_s, C = X_seq.shape
    X_comb = np.hstack([X_w.reshape(Nw, T_w),
                        X_f,
                        X_seq.reshape(N_, T_s * C)])
    try:
        counts = np.bincount(y)
        min_c  = counts[counts > 0].min()
        k_nn   = max(1, min(5, min_c - 1))
        X_sm, y_sm = SMOTE(random_state=SEED, k_neighbors=k_nn).fit_resample(X_comb, y)
    except Exception as e:
        print(f"  [SMOTE warn] {e} — skipping")
        return X_w, X_f, X_seq, y
    X_w_sm   = X_sm[:, :T_w].reshape(-1, T_w, 1)
    X_f_sm   = X_sm[:, T_w : T_w + X_f.shape[1]]
    X_seq_sm = X_sm[:, T_w + X_f.shape[1]:].reshape(-1, T_s, C)
    return X_w_sm, X_f_sm, X_seq_sm, y_sm

print("\n  Applying SMOTE on RAW data ...")
X_w_tr_raw_sm, X_f_tr_raw_sm, X_seq_tr_raw_sm, y_tr_raw_sm = apply_smote_multi(
    X_w_tr_z, X_f_tr_s, X_seq_tr_s, y_tr_enc)
print(f"  Train after SMOTE (RAW): {len(y_tr_raw_sm)}")
print(f"  Class dist (RAW): {dict(zip(*np.unique(y_tr_raw_sm, return_counts=True)))}")

class AttentionLayer(layers.Layer):
    def __init__(self, units=64, **kw):
        super().__init__(**kw)
        self.units = units
    def build(self, input_shape):
        D = input_shape[-1]
        self.W = self.add_weight(shape=(D, self.units),
                                 initializer='glorot_uniform', name='W')
        self.b = self.add_weight(shape=(self.units,),
                                 initializer='zeros', name='b')
        self.u = self.add_weight(shape=(self.units, 1),
                                 initializer='glorot_uniform', name='u')
    def call(self, x):
        uit = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        ait = tf.squeeze(tf.tensordot(uit, self.u, axes=1), -1)
        a   = tf.expand_dims(tf.nn.softmax(ait, axis=1), -1)
        return tf.reduce_sum(x * a, axis=1)
    def get_config(self):
        cfg = super().get_config()
        cfg.update({'units': self.units})
        return cfg


def build_multi_input_classifier(beat_len, n_feat, seq_len, seq_chan, n_classes):
    inp_w = Input(shape=(beat_len, 1), name='wave_input')
    a = layers.Bidirectional(
            layers.LSTM(LSTM_UNITS, return_sequences=True,
                        dropout=LSTM_DROPOUT,
                        kernel_regularizer=tf.keras.regularizers.l2(1e-4)))(inp_w)
    a = layers.LayerNormalization()(a)
    a = layers.Bidirectional(
            layers.LSTM(LSTM_UNITS, return_sequences=True, dropout=LSTM_DROPOUT))(a)
    a = layers.LayerNormalization()(a)
    a = AttentionLayer(units=LSTM_UNITS)(a)

    inp_seq = Input(shape=(seq_len, seq_chan), name='rr_seq_input')
    b = layers.Bidirectional(
            layers.LSTM(32, return_sequences=True, dropout=LSTM_DROPOUT))(inp_seq)
    b = layers.Bidirectional(
            layers.LSTM(16, return_sequences=False, dropout=LSTM_DROPOUT))(b)

    inp_f = Input(shape=(n_feat,), name='feat_input')
    c = layers.Dense(64, activation='relu',
                     kernel_regularizer=tf.keras.regularizers.l2(1e-4))(inp_f)
    c = layers.Dropout(LSTM_DROPOUT)(c)
    c = layers.Dense(32, activation='relu')(c)

    z = layers.Concatenate()([a, b, c])
    z = layers.Dense(64, activation='relu')(z)
    z = layers.Dropout(LSTM_DROPOUT)(z)
    z = layers.Dense(32, activation='relu')(z)
    out = layers.Dense(n_classes, activation='softmax')(z)

    model = Model(inputs=[inp_w, inp_seq, inp_f], outputs=out,
                  name='RAW_BiLSTM')
    model.compile(optimizer=optimizers.Adam(LSTM_LR),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def evaluate_predictions(y_true, y_pred, y_proba, n_cls):
    m = {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, average='macro', zero_division=0),
        'recall'   : recall_score(y_true, y_pred, average='macro', zero_division=0),
        'f1'       : f1_score(y_true, y_pred, average='macro', zero_division=0),
        'f2'       : fbeta_score(y_true, y_pred, average='macro', beta=2, zero_division=0),
    }
    try:
        yt_bin = label_binarize(y_true, classes=np.arange(n_cls))
        valid  = yt_bin.sum(axis=0) > 0
        m['auc_roc'] = (roc_auc_score(yt_bin[:, valid], y_proba[:, valid],
                                       multi_class='ovr', average='macro')
                        if valid.sum() > 1 else float('nan'))
    except Exception:
        m['auc_roc'] = float('nan')
    return m


tf.keras.backend.clear_session()
classifier_p1 = build_multi_input_classifier(
    beat_len=BEAT_LEN, n_feat=X_f_tr_raw_sm.shape[1],
    seq_len=SEQ_LEN, seq_chan=3, n_classes=N_CLASSES,
)
print("\n  RAW classifier summary:")
classifier_p1.summary(print_fn=lambda t: print('    ' + t))

cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_tr_raw_sm)
class_weight_dict = {i: w for i, w in enumerate(cw)}

cb1 = [
    callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6),
]

print(f"\n  Training Phase 1 ({LSTM_EPOCHS} epochs max) ...")
hist_p1 = classifier_p1.fit(
    [X_w_tr_raw_sm, X_seq_tr_raw_sm, X_f_tr_raw_sm],
    to_categorical(y_tr_raw_sm, N_CLASSES),
    validation_split=0.15, epochs=LSTM_EPOCHS, batch_size=LSTM_BATCH,
    callbacks=cb1, class_weight=class_weight_dict, verbose=2)

yprob_p1 = classifier_p1.predict(
    [X_w_te_z, X_seq_te_s, X_f_te_s],
    batch_size=LSTM_BATCH, verbose=0)
yp_p1 = yprob_p1.argmax(axis=1)

m_p1 = evaluate_predictions(y_te_enc, yp_p1, yprob_p1, N_CLASSES)
m_p1.update({'phase': 'P1_RAW_LSTM', 'y_true': y_te_enc,
             'y_pred': yp_p1, 'y_proba': yprob_p1, 'history': hist_p1.history})
print(f"\n  [P1_RAW_LSTM]  acc={m_p1['accuracy']:.4f}  F1={m_p1['f1']:.4f}  "
      f"F2={m_p1['f2']:.4f}  AUC={m_p1['auc_roc']:.4f}")

classifier_p1.save(f'{OUT_DIR}/p1_raw_lstm_classifier.keras')


# =============================================================================
# 9.5 SPECTRAL GATING DENOISER
# =============================================================================
# This block follows the style of experiment_ecg_stl.ipynb.
# It replaces the STL denoising module by STFT-based Spectral Gating.
#
# Phase 2: beat-level spectral gating
#          preprocessed ECG -> segmentation -> spectral gate each beat
#
# Phase 3: record-level spectral gating
#          preprocessed ECG -> spectral gate full record -> segmentation
#
# The downstream MultiInput BiLSTM + Attention classifier is unchanged.

print("\n" + "="*72)
print("  SPECTRAL GATING DENOISING MODULE")
print("="*72)

SPECTRAL_RECORD_NPERSEG = 256
SPECTRAL_RECORD_NOVERLAP = 192
SPECTRAL_BEAT_NPERSEG = 64
SPECTRAL_BEAT_NOVERLAP = 48
SPECTRAL_NOISE_PERCENTILE = 25
SPECTRAL_GATE_STRENGTH = 1.15
SPECTRAL_MASK_FLOOR = 0.05
EPS_SPEC = 1e-12

SPECTRAL_CACHE_DIR = os.path.join(OUT_DIR, "spectral_cache")
os.makedirs(SPECTRAL_CACHE_DIR, exist_ok=True)


def spectral_soft_gate_1d(x, nperseg=256, noverlap=192, fs=FS,
                          noise_percentile=SPECTRAL_NOISE_PERCENTILE,
                          gate_strength=SPECTRAL_GATE_STRENGTH,
                          mask_floor=SPECTRAL_MASK_FLOOR):
    """
    STFT soft spectral gating for one 1D signal.

    The method estimates a frequency-wise noise floor from the STFT magnitude,
    applies a soft mask, and reconstructs the denoised signal by iSTFT.
    """
    x = np.asarray(x, dtype=float).reshape(-1)
    if len(x) < 4:
        return x.astype(np.float32)

    nper = min(int(nperseg), len(x))
    nover = min(int(noverlap), max(0, nper - 1))

    _, _, Z = stft(x, fs=fs, nperseg=nper, noverlap=nover,
                   boundary="zeros", padded=True)

    mag = np.abs(Z)
    noise_floor = np.percentile(mag, noise_percentile, axis=1, keepdims=True)

    mask = 1.0 - gate_strength * noise_floor / (mag + EPS_SPEC)
    mask = np.clip(mask, mask_floor, 1.0)

    _, rec = istft(Z * mask, fs=fs, nperseg=nper, noverlap=nover,
                   input_onesided=True, boundary=True)

    if len(rec) < len(x):
        rec = np.pad(rec, (0, len(x) - len(rec)), mode="edge")
    return rec[:len(x)].astype(np.float32)


def spectral_denoise_beat(beat):
    return spectral_soft_gate_1d(
        beat,
        nperseg=SPECTRAL_BEAT_NPERSEG,
        noverlap=SPECTRAL_BEAT_NOVERLAP,
        fs=FS,
    )


def spectral_denoise_record(sig_clean):
    return spectral_soft_gate_1d(
        sig_clean,
        nperseg=SPECTRAL_RECORD_NPERSEG,
        noverlap=SPECTRAL_RECORD_NOVERLAP,
        fs=FS,
    )


def _cache_path_for_spectral_record(rid):
    return os.path.join(SPECTRAL_CACHE_DIR, f"record_{rid}_spectral.npy")


def get_or_create_spectral_record(rid, sig_clean):
    """
    Cache record-level spectral denoising because it is reused across runs.
    """
    path = _cache_path_for_spectral_record(rid)
    if os.path.exists(path):
        try:
            cached = np.load(path)
            if len(cached) == len(sig_clean):
                return cached.astype(np.float32)
        except Exception:
            pass

    den = spectral_denoise_record(sig_clean)
    np.save(path, den.astype(np.float32))
    return den.astype(np.float32)


def build_per_patient_beats_spectral_beat_level(rid):
    """
    Beat-level spectral gating:
      load record -> median+bandpass -> segment beats -> denoise each beat.
    """
    try:
        mlii, beats_ann, sig_len = load_record(rid)
    except FileNotFoundError:
        return []
    except Exception as e:
        print(f"  [WARN] {rid}: {e}")
        return []

    sig_clean = preprocess_signal(mlii)

    r_positions, labels = [], []
    for sample_pos, beat_type in beats_ann:
        if beat_type not in LABEL_MAP:
            continue
        if sample_pos - WB < 0 or sample_pos + WA >= sig_len:
            continue
        r_positions.append(sample_pos)
        labels.append(LABEL_MAP[beat_type])

    r_positions = np.array(r_positions)
    if len(r_positions) < 2:
        return []

    RR, BPM, RR_z = compute_rr_bpm_zscore(r_positions)
    out = []

    for i, (s, lab) in enumerate(zip(r_positions, labels)):
        raw_beat = sig_clean[s - WB : s + WA]
        den_beat = spectral_denoise_beat(raw_beat)
        feat = extract_handcrafted(den_beat, RR[i], BPM[i], RR_z[i])
        out.append({
            "beat": den_beat.astype(np.float32),
            "rr": float(RR[i]),
            "bpm": float(BPM[i]),
            "rr_z": float(RR_z[i]),
            "feat": feat,
            "label": lab,
            "patient": rid,
        })

    return out


def build_per_patient_beats_spectral_record_level(rid):
    """
    Record-level spectral gating:
      load record -> median+bandpass -> spectral gate full record -> segment beats.
    """
    try:
        mlii, beats_ann, sig_len = load_record(rid)
    except FileNotFoundError:
        return []
    except Exception as e:
        print(f"  [WARN] {rid}: {e}")
        return []

    sig_clean = preprocess_signal(mlii)
    sig_spec = get_or_create_spectral_record(rid, sig_clean)

    r_positions, labels = [], []
    for sample_pos, beat_type in beats_ann:
        if beat_type not in LABEL_MAP:
            continue
        if sample_pos - WB < 0 or sample_pos + WA >= sig_len:
            continue
        r_positions.append(sample_pos)
        labels.append(LABEL_MAP[beat_type])

    r_positions = np.array(r_positions)
    if len(r_positions) < 2:
        return []

    RR, BPM, RR_z = compute_rr_bpm_zscore(r_positions)
    out = []

    for i, (s, lab) in enumerate(zip(r_positions, labels)):
        beat = sig_spec[s - WB : s + WA]
        feat = extract_handcrafted(beat, RR[i], BPM[i], RR_z[i])
        out.append({
            "beat": beat.astype(np.float32),
            "rr": float(RR[i]),
            "bpm": float(BPM[i]),
            "rr_z": float(RR_z[i]),
            "feat": feat,
            "label": lab,
            "patient": rid,
        })

    return out


def build_dataset_spectral(builder_fn, phase_name):
    train_beats_sp, test_beats_sp = [], []

    print(f"\n  {phase_name}: loading TRAIN patients ...")
    for rid in tqdm(TRAIN_PATIENTS):
        train_beats_sp += builder_fn(rid)

    print(f"\n  {phase_name}: loading TEST patients ...")
    for rid in tqdm(TEST_PATIENTS):
        test_beats_sp += builder_fn(rid)

    return train_beats_sp, test_beats_sp


def prepare_spectral_arrays(train_beats_sp, test_beats_sp):
    X_w_tr_sp, X_f_tr_sp, X_rr_tr_sp, y_tr_raw_sp, pid_tr_sp = beats_to_arrays(train_beats_sp)
    X_w_te_sp, X_f_te_sp, X_rr_te_sp, y_te_raw_sp, pid_te_sp = beats_to_arrays(test_beats_sp)

    y_tr_enc_sp = le.transform(y_tr_raw_sp)
    y_te_enc_sp = le.transform(y_te_raw_sp)

    X_w_tr_sp = zscore_waveform(X_w_tr_sp)
    X_w_te_sp = zscore_waveform(X_w_te_sp)

    sc_f_sp = StandardScaler()
    X_f_tr_sp_s = sc_f_sp.fit_transform(X_f_tr_sp)
    X_f_te_sp_s = sc_f_sp.transform(X_f_te_sp)

    X_seq_tr_sp = build_rr_sequences(X_rr_tr_sp, pid_tr_sp)
    X_seq_te_sp = build_rr_sequences(X_rr_te_sp, pid_te_sp)

    sc_rr_sp = StandardScaler()
    Ntr, Ts, Cs = X_seq_tr_sp.shape
    X_seq_tr_sp_s = sc_rr_sp.fit_transform(X_seq_tr_sp.reshape(-1, Cs)).reshape(Ntr, Ts, Cs)
    X_seq_te_sp_s = sc_rr_sp.transform(X_seq_te_sp.reshape(-1, Cs)).reshape(len(X_seq_te_sp), Ts, Cs)

    return (X_w_tr_sp, X_f_tr_sp_s, X_seq_tr_sp_s, y_tr_enc_sp,
            X_w_te_sp, X_f_te_sp_s, X_seq_te_sp_s, y_te_enc_sp)


def train_spectral_phase(train_beats_sp, test_beats_sp, phase_name):
    """
    Train the same MultiInput BiLSTM + Attention classifier on denoised beats.
    """
    (X_w_tr_sp, X_f_tr_sp_s, X_seq_tr_sp_s, y_tr_enc_sp,
     X_w_te_sp, X_f_te_sp_s, X_seq_te_sp_s, y_te_enc_sp) = prepare_spectral_arrays(
        train_beats_sp, test_beats_sp
    )

    print(f"  {phase_name} train shape: wave={X_w_tr_sp.shape}, feat={X_f_tr_sp_s.shape}, seq={X_seq_tr_sp_s.shape}")
    print(f"  {phase_name} test  shape: wave={X_w_te_sp.shape}, feat={X_f_te_sp_s.shape}, seq={X_seq_te_sp_s.shape}")

    print(f"\n  Applying SMOTE on {phase_name} data ...")
    X_w_tr_sm, X_f_tr_sm, X_seq_tr_sm, y_tr_sm = apply_smote_multi(
        X_w_tr_sp, X_f_tr_sp_s, X_seq_tr_sp_s, y_tr_enc_sp
    )
    print(f"  Train after SMOTE: {len(y_tr_sm)}")
    print(f"  Class dist: {dict(zip(*np.unique(y_tr_sm, return_counts=True)))}")

    tf.keras.backend.clear_session()
    model = build_multi_input_classifier(
        beat_len=BEAT_LEN,
        n_feat=X_f_tr_sm.shape[1],
        seq_len=SEQ_LEN,
        seq_chan=3,
        n_classes=N_CLASSES,
    )

    cw_phase = compute_class_weight("balanced", classes=np.arange(N_CLASSES), y=y_tr_sm)
    cw_phase = {i: w for i, w in enumerate(cw_phase)}

    cb = [
        callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    ]

    print(f"\n  Training {phase_name} ({LSTM_EPOCHS} epochs max) ...")
    hist = model.fit(
        [X_w_tr_sm, X_seq_tr_sm, X_f_tr_sm],
        to_categorical(y_tr_sm, N_CLASSES),
        validation_split=0.15,
        epochs=LSTM_EPOCHS,
        batch_size=LSTM_BATCH,
        callbacks=cb,
        class_weight=cw_phase,
        verbose=2,
    )

    yprob = model.predict(
        [X_w_te_sp, X_seq_te_sp_s, X_f_te_sp_s],
        batch_size=LSTM_BATCH,
        verbose=0,
    )
    ypred = yprob.argmax(axis=1)

    metrics = evaluate_predictions(y_te_enc_sp, ypred, yprob, N_CLASSES)
    metrics.update({
        "phase": phase_name,
        "y_true": y_te_enc_sp,
        "y_pred": ypred,
        "y_proba": yprob,
        "history": hist.history,
    })

    print(
        f"\n  [{phase_name}] acc={metrics['accuracy']:.4f} "
        f"F1={metrics['f1']:.4f} F2={metrics['f2']:.4f} AUC={metrics['auc_roc']:.4f}"
    )

    model.save(f"{OUT_DIR}/{phase_name.lower()}_classifier.keras")
    return metrics, model


# =============================================================================
# 10. PHASE 2 — BEAT-LEVEL SPECTRAL GATING
# =============================================================================
train_beats_spec_beat, test_beats_spec_beat = build_dataset_spectral(
    build_per_patient_beats_spectral_beat_level,
    "P2_Spectral_Beat_Level",
)
m_p2, classifier_p2 = train_spectral_phase(
    train_beats_spec_beat,
    test_beats_spec_beat,
    "P2_SPECTRAL_BEAT_LEVEL",
)


# =============================================================================
# 11. PHASE 3 — RECORD-LEVEL SPECTRAL GATING
# =============================================================================
train_beats_spec_record, test_beats_spec_record = build_dataset_spectral(
    build_per_patient_beats_spectral_record_level,
    "P3_Spectral_Record_Level",
)
m_p3, classifier_p3 = train_spectral_phase(
    train_beats_spec_record,
    test_beats_spec_record,
    "P3_SPECTRAL_RECORD_LEVEL",
)


# =============================================================================
# 12. SUMMARY TABLE
# =============================================================================
all_phases = [m_p1, m_p2, m_p3]

rows = []
for m in all_phases:
    rows.append({
        "phase": m["phase"],
        "accuracy": m["accuracy"],
        "precision": m["precision"],
        "recall": m["recall"],
        "f1": m["f1"],
        "f2": m["f2"],
        "auc_roc": m["auc_roc"],
    })

df_summary = pd.DataFrame(rows)
print("\n" + "="*72)
print("  ECG SPECTRAL SUMMARY")
print("="*72)
print(df_summary.to_string(index=False))
df_summary.to_csv(f"{OUT_DIR}/ecg_spectral_summary.csv", index=False)

df_paper = df_summary[["phase", "accuracy", "precision", "recall", "f1"]].copy()
df_paper.columns = ["Method", "Acc.", "Prec.", "Rec.", "F1"]
df_paper[["Acc.", "Prec.", "Rec.", "F1"]] = df_paper[["Acc.", "Prec.", "Rec.", "F1"]].round(4)
df_paper.to_csv(f"{OUT_DIR}/ecg_spectral_paper_table.csv", index=False)


# =============================================================================
# 13. VISUALIZATIONS
# =============================================================================
plt.figure(figsize=(10, 5))
x = np.arange(len(df_summary))
plt.bar(x - 0.18, df_summary["accuracy"], width=0.36, label="Accuracy")
plt.bar(x + 0.18, df_summary["f1"], width=0.36, label="F1 macro")
plt.xticks(x, df_summary["phase"], rotation=15, ha="right")
plt.ylim(0, 1.05)
plt.title("MIT-BIH ECG — Spectral Gating Experiment")
plt.ylabel("Score")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ecg_spectral_metrics.png", dpi=200)
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, m in zip(axes, all_phases):
    cm = confusion_matrix(m["y_true"], m["y_pred"], labels=np.arange(N_CLASSES))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES_ORDER, yticklabels=CLASSES_ORDER, ax=ax, cbar=False)
    ax.set_title(f"{m['phase']}\nF1={m['f1']:.3f}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/ecg_spectral_confusion.png", dpi=200)
plt.close()


# =============================================================================
# 14. CLASSIFICATION REPORTS + PER-CLASS CSV
# =============================================================================
report_rows = []
for m in all_phases:
    rep = classification_report(
        m["y_true"], m["y_pred"], labels=np.arange(N_CLASSES),
        target_names=CLASSES_ORDER, output_dict=True, zero_division=0,
    )
    for cname in CLASSES_ORDER:
        report_rows.append({
            "phase": m["phase"],
            "class": cname,
            "precision": rep[cname]["precision"],
            "recall": rep[cname]["recall"],
            "f1": rep[cname]["f1-score"],
            "support": rep[cname]["support"],
        })

pd.DataFrame(report_rows).to_csv(f"{OUT_DIR}/ecg_spectral_perclass_metrics.csv", index=False)


# =============================================================================
# 15. FINAL VERDICT
# =============================================================================
best = max(all_phases, key=lambda m: m["f1"])
print(f"\n{'='*72}")
print(f"  BEST PHASE : {best['phase']}")
print(f"  Accuracy   = {best['accuracy']:.4f}")
print(f"  F1 macro   = {best['f1']:.4f}")
print(f"  F2 macro   = {best['f2']:.4f}")
print(f"  AUC-ROC    = {best['auc_roc']:.4f}")
print("\n  Spectral gains over baseline:")
print(f"    Beat-level  ΔF1  = {m_p2['f1'] - m_p1['f1']:+.4f}")
print(f"    Record-level ΔF1 = {m_p3['f1'] - m_p1['f1']:+.4f}")
print(f"    Record-level ΔAcc = {m_p3['accuracy'] - m_p1['accuracy']:+.4f}")
print(f"\n  All outputs → {OUT_DIR}/")
print(f"{'='*72}")
